In [11]:
import os
import shutil
import sys

# Ensure local modules can be imported
sys.path.append(os.getcwd())

from active_learning.learner import ActiveLearner
from active_learning.storage import ActiveLearningLogger

# 1. Setup Test Environment
TEST_STORAGE_DIR = "test_active_learning_data"
TEST_MODEL_FILE = "test_model_weights.json"

# Clean up previous runs
if os.path.exists(TEST_STORAGE_DIR):
    shutil.rmtree(TEST_STORAGE_DIR)
os.makedirs(TEST_STORAGE_DIR, exist_ok=True)

print("Test environment setup complete.")

Test environment setup complete.


In [12]:
# 2. Simulate User Feedback Generation

logger = ActiveLearningLogger(storage_dir=TEST_STORAGE_DIR)

# Scenario: User gates CellLineA
# Live Gate (5 vertices)
logger.log_gate_feedback(
    sample_name="Sample_A1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="SSC-A",
    gate_type="Polygon",
    gate_params={'vertices': [[1, 1], [2, 1], [3, 2], [2, 3], [1, 2]]}, # 5 points = Live
    user_decision="accepted"
)

# Singlet Gate (4 vertices)
logger.log_gate_feedback(
    sample_name="Sample_A1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="FSC-H",
    gate_type="Polygon",
    gate_params={'vertices': [[10, 10], [20, 10], [20, 20], [10, 20]]}, # 4 points = Singlet
    user_decision="accepted"
)

# Scenario: User gates CellLineB (Different cell line)
# Live Gate (5 vertices)
logger.log_gate_feedback(
    sample_name="Sample_B1",
    cell_line=" CellLineB ", # Note extra spaces to test normalization
    x_channel="FSC-A",
    y_channel="SSC-A",
    gate_type="Polygon",
    gate_params={'vertices': [[5, 5], [6, 5], [7, 6], [6, 7], [5, 6]]},
    user_decision="manual_override"
)

print("Simulated user feedback logged.")

Simulated user feedback logged.


In [9]:
# 3. specific Active Learner Training

learner = ActiveLearner(storage_dir=TEST_STORAGE_DIR, model_filename=TEST_MODEL_FILE)

# Initial state should be empty or default
print("Known cell lines before training:", learner.get_known_cell_lines())

# Train (Process logs)
learner.train()

print("Known cell lines after training:", learner.get_known_cell_lines())

# Verify Predictions
# CellLineA
pred_live_a = learner.predict("CellLineA", "Live")
pred_singlet_a = learner.predict("CellLineA", "Singlet")

print(f"\nCellLineA Live Gate (Expected 5 pts): {len(pred_live_a) if pred_live_a else 'None'}")
print(f"CellLineA Singlet Gate (Expected 4 pts): {len(pred_singlet_a) if pred_singlet_a else 'None'}")

# CellLineB (Should be normalized)
pred_live_b = learner.predict("CellLineB", "Live")
print(f"CellLineB Live Gate (Expected 5 pts): {len(pred_live_b) if pred_live_b else 'None'}")

# CellLineC (Unknown)
pred_live_c = learner.predict("CellLineC", "Live")
print(f"CellLineC Live Gate (Expected None): {pred_live_c}")

No existing model weights found. Starting fresh.
Known cell lines before training: []
Training complete. Knowledge base updated for 2 cell lines.
Model weights saved to test_active_learning_data/test_model_weights.json
Known cell lines after training: ['CellLineA', 'CellLineB']

CellLineA Live Gate (Expected 5 pts): 5
CellLineA Singlet Gate (Expected 4 pts): 4
CellLineB Live Gate (Expected 5 pts): 5
CellLineC Live Gate (Expected None): None


In [10]:
# 4. Persistence Test (Load from disk)

# Create a new instance to simulate restarting the application
learner_new = ActiveLearner(storage_dir=TEST_STORAGE_DIR, model_filename=TEST_MODEL_FILE)

print("Loaded known cell lines from disk:", learner_new.get_known_cell_lines())

assert "CellLineA" in learner_new.get_known_cell_lines()
assert "CellLineB" in learner_new.get_known_cell_lines()

print("\nTest passed successfully!")

Model weights loaded from test_active_learning_data/test_model_weights.json
Loaded known cell lines from disk: ['CellLineA', 'CellLineB']

Test passed successfully!
